# D-NCQAS revision 3: execute and verify

Run these cells in order for a fresh smoke check and the numerical tests. The complete revised research results are bundled separately under outputs/analysis. Full research reproduction uses the commands in QUICKSTART.md. The notebook does not rerun the older publication configuration as if it were the revised study.

In [ ]:
import os, sys, json, zipfile, subprocess, importlib.util
from pathlib import Path
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
# Locate an extracted project first; do not guess an arbitrary ZIP.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "D-NCQAS"]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "configs/smoke.json").is_file() and (p / "src/d_ncqas").is_dir()), None)
if PROJECT_ROOT is None:
    archive = Path("D_NCQAS_Revision3_Project.zip")
    if not archive.exists() and IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one complete project ZIP")
        archive = Path(next(iter(uploaded)))
    if not archive.exists():
        raise FileNotFoundError("Extract the complete project and run from its root, or upload D_NCQAS_Revision3_Project.zip")
    destination = Path.cwd() / "d_ncqas_extracted"
    with zipfile.ZipFile(archive) as z:
        if "D-NCQAS/configs/smoke.json" not in z.namelist():
            raise ValueError("This is not the complete revised project archive")
        for member in z.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive path")
        z.extractall(destination)
    PROJECT_ROOT = (destination / "D-NCQAS").resolve()
print("Project:", PROJECT_ROOT)
# Install before importing NumPy/SciPy/pandas. A skipped install is only for a known prepared environment.
INSTALL_DEPENDENCIES = os.environ.get("D_NCQAS_SKIP_INSTALL") != "1"
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements-runtime.lock.txt")], check=True)
print("Dependencies ready")

In [ ]:
import numpy as np, pandas as pd, scipy
print("Python", sys.version.split()[0], "NumPy", np.__version__, "SciPy", scipy.__version__)
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=PROJECT_ROOT, text=True, capture_output=True)
print(result.stdout + result.stderr)
result.check_returncode()

In [ ]:
import uuid
config_name = "smoke.json"  # Research reproduction is documented in QUICKSTART.md
RUN_DIR = PROJECT_ROOT / "outputs" / ("notebook_" + uuid.uuid4().hex[:10])
def run_script(name, *args):
    result = subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / name), *map(str, args)], cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr[-5000:])
    result.check_returncode()
run_script("run_all_experiments.py", "--config", PROJECT_ROOT / "configs" / config_name, "--output", RUN_DIR)
run_script("generate_figures_and_tables.py", "--output", RUN_DIR)
run_script("verify_pipeline_integrity.py", "--output", RUN_DIR)
print("Completed:", RUN_DIR)

In [ ]:
from IPython.display import Image, display
summary = pd.read_csv(RUN_DIR / "processed/ghz_summary_table.csv")
display(summary[summary.regime == "ou"])
display(pd.read_csv(RUN_DIR / "tables/statistical_tests.csv"))
display(Image(filename=str(RUN_DIR / "figures/fig1_ghz_robustness.png"), width=1100))
print("Smoke results validate execution only; do not use them as adequately powered publication evidence.")
research = PROJECT_ROOT / "outputs/analysis/tables/primary_tests.csv"
if research.exists():
    print("Bundled revision-3 primary comparisons (separate from this smoke run):")
    display(pd.read_csv(research))
else:
    print("See the full project/output archive for the completed revision-3 research tables.")

In [ ]:
output_zip = PROJECT_ROOT / (RUN_DIR.name + ".zip")
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RUN_DIR.rglob("*")):
        if p.is_file(): z.write(p, Path(RUN_DIR.name) / p.relative_to(RUN_DIR))
print("Output archive:", output_zip)
if IN_COLAB:
    from google.colab import files
    files.download(str(output_zip))